# Multivariate modelling with VGP

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude #--force-reinstall

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from cmcrameri import cm

import geoml

import geoml.kernels as kr
import geoml.transform as tr
import geoml.likelihood as lk
import geoml.latent as gl
import geoml.warping as wp

The Jura dataset:

In [ ]:
jura_train, jura_validation = geoml.datasets.jura()

ELEMENTS = jura_train.variables['Elements'].labels

jura_train

## Metal contamination

There are 7 variables representing heavy metal contamination, presenting some non-linear correlations and extreme values. The variational Gaussian process (VGP) model will seek to recognize this pattern.



### Step #0: exploratory data analysis

It is important to do some exploratory work before modelling, because the structure of the data helps with the modelling decisions later.

In [ ]:
exp_data = geoml.plots.Explorer(jura_train, continuous='Elements')
exp_data.pairs(log=True, principal_components=3, upper='density');

The Princpal Components Analysis (PCA) reveals the correlation structure of the variables.

In [ ]:
exp_data.pca(0.95, log=True);

### Step #1: inducing points

The VGP requires a set of *inducing points*. In this example we will use a combination of a regular grid of points and the data points.

In [ ]:
inducing_points = geoml.data.inducing.experts(
    geoml.data.inducing.combine(
        jura_train,
        geoml.data.Grid2D(start=[0, 0], n=[21, 21], end=[6, 6])),
    4)

### Step #2: building a latent variable network

The VGP is based on the concept of *latent variables*, which can be combined in different ways to fit the data, forming a network. It starts with an input node (the spatial coordinates) and, in this case, two different, stationary latent variables are linearly combined in space, then its components are linearly mixed to form the correlated latent variables that will be linked to the data variables.

In [ ]:
# network input node
net_input = gl.BasicInput(inducing_points, tr.Isotropic(1))

# output
net_output = gl.BasicGP(net_input, size=7, kernel=kr.Spherical())

### Step #3: likelihoods and normalization

The likelihood (distribution of the error) must be defined. In this case the $\epsilon$-insensitive likelihood is used to deal with the extreme values.

The likelihood includes the *warping function*, used to normalize the data to zero mean and unit variance, deal with the non-negativity constraint (these are metal concentrations) and to compensate for asymmetric distributions (see the histograms below).

In [ ]:
warp = wp.ChainedWarping(
    wp.Log(7),                   # non-negativity
    wp.ZScore(7),                     # centering / scaling
    wp.RobustPCA(7, 7),               # decorrelation
    wp.Spline(7, knots_per_arm=5),    # asymmetry
    wp.ZScore(7),
)

likelihood = lk.EpsilonInsensitive(warp)

### Step #4: training

The VGP model is created and trained. It works by maximizing the Evidence Lower Bound (ELBO).

In [ ]:
model = geoml.models.VGPNetwork(
    data=jura_train,
    variables='Elements',
    likelihoods=likelihood,
    latent_network=net_output,
    options=geoml.models.GPOptions(prediction_batch_size=2000))
model.set_learning_rate(5e-2)
model.train_full(100)

### Step #5: diagnostics

The model is predicted onto the data points, to compare the measurements with the fitted points.

In [ ]:
model.predict(jura_train, n_sim=100)
jura_train.variables['Elements'].reset_quantiles([0.025, 0.5, 0.975])

The `Explorer` class process the data into useful forms for plotting.

In [ ]:
exp_data = geoml.plots.Explorer(jura_train, continuous='Elements', model=model)

The training curve allows checking the model for convergence.

In [ ]:
exp_data.training_curve();

The accuracy plot evaluates the predictions confidence intervals. A model that envelops the data properly has a 1:1 curve.

In [ ]:
exp_data.accuracy();

The spread plot checks if the model's local uncertainty is proportional to the variability in the data. As natural variables are often heteroscedastic, the variance is not constant with respect to the absolute value.

In [ ]:
exp_data.spread_check();

A pairs plot in transformed space reveals if the model was able to normalize and decorrelate the data.

In [ ]:
exp_data.transformed_pairs(upper='density');

With the simulations at the data locations it is possible to see if the data distribution was captured.

In [ ]:
exp_data.simulation_pairs();

## Prediction in a grid and plotting

The noise in the data is automatically integrated, so the individual simulations are smooth with respect to the fitted kernel.

In [ ]:
# creating empty grid
grid = geoml.data.Grid2D(start=[0, 0], n=[251, 251], end=[6, 6])

# prediction (with simulations)
model.predict(grid, n_sim=100, include_noise=True)

The quantiles and predictions are derived from the samples, and are thus noisy due to the finite population. The `sigma` parameter is used to smooth the results (especially the distribution tails) for plotting.

In [ ]:
# quantiles of each element's predictive distribution
grid.variables['Elements'].reset_quantiles([0.025, 0.5, 0.975])
quant = {v: grid.variables['Elements'].components[v].quantiles for v in ELEMENTS}

Prediction (median, confidence interval, and model fit):

In [ ]:
maxs = [np.max(jura_train.variables['Elements'].components[el].measurements.values)
        for el in ELEMENTS]
maxs = np.array(maxs)
sigma = 3

fig, ax = plt.subplots(7, 5, figsize=(12, 20),
                       gridspec_kw={"width_ratios": [1, 1, 1, 0.1, 1],
                                    "hspace": 0.3, "wspace": 0.1})
for i, elem in enumerate(ELEMENTS):
    ax[i, 0].scatter(
        jura_train.coordinates[:, 0],
        jura_train.coordinates[:, 1],
        c=jura_train.variables['Elements'].components[elem].measurements.values,
        cmap=cm.davos, s=2)
    ax[i, 0].set_xlim(0, 6)
    ax[i, 0].set_ylim(0, 6)
    ax[i, 0].set_aspect("equal")
    ax[i, 0].set_title(elem + " - data")
    im = ax[i, 1].imshow(
        quant[elem][0.5].as_image(sigma=None),
        vmin=0, vmax=maxs[i],
        origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
    ax[i, 1].set_title(elem + " - median")
    ax[i, 1].set_xticks([0, 2, 4, 6])
    ax[i, 2].imshow(
        quant[elem][0.975].as_image(sigma=sigma) - quant[elem][0.025].as_image(sigma=sigma),
        # quant[elem][0.975] - quant[elem][0.025],
        vmin=0, vmax=maxs[i],
        origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
    ax[i, 2].set_title(elem + " - 95% C.I. width")
    ax[i, 2].set_xticks([0, 2, 4, 6])

    plt.colorbar(im, cax=ax[i, 3])
    ax[i, 3].set_aspect(10, adjustable="box")
    box = ax[i, 3].get_position()
    box.p0[0] -= 0.02
    ax[i, 3].set_position(box)

    ax[i, 4].plot([0, maxs[i]], [0, maxs[i]], "-k")
    ax[i, 4].scatter(
        jura_train.variables['Elements'].components[elem].measurements.values,
        jura_train.variables['Elements'].components[elem].quantiles[0.5].values,
        alpha=0.2)
    ax[i, 4].set_title(elem + " - real vs. predicted")
    ax[i, 4].set_aspect("equal")
    ax[i, 4].set_xlim([0, maxs[i]])
    ax[i, 4].set_ylim([0, maxs[i]])
    ax[i, 4].set_yticklabels([" "]*len(ax[i, 4].get_yticklabels()))

fig.show()

Simulations:

In [ ]:
fig, ax = plt.subplots(7, 6, figsize=(12, 18),
                       gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 0.1],
                                    "hspace": 0.2, "wspace": 0.3})
for i, elem in enumerate(ELEMENTS):
    for j in range(5):
        im = ax[i, j].imshow(
            grid.variables['Elements'].components[elem].simulation(j).as_image(sigma=sigma),
            vmin=0, vmax=maxs[i],
            origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
        ax[i, j].set_title(elem + " - simulation %d" % j)
        ax[i, j].set_xticks([0, 2, 4, 6])

        plt.colorbar(im, cax=ax[i, 5])
        ax[i, 5].set_aspect(10, adjustable="box")

fig.show()